# Momentum + Multi-Strategy Edge Research v1.1 — Continual Dataset Safe

Notebook ini adalah revisi dari `Momentum_MultiStrategy_Edge_Research_Notebook_v1`.

Perubahan utama v1.1:

1. Default dataset diarahkan ke continual dataset:
   `data/datasets/continual/q3_2024_to_2026_04_30/full_labeled.parquet`
2. Target configuration dibuat robust:
   - tetap berjalan walaupun `score_cols` kosong,
   - tetap berjalan walaupun sebagian `fwd_ret_*` missing,
   - tidak membuat `DataFrame([]).sort_values(["target", ...])` error.
3. Top-k edge evaluation mengevaluasi **signature score manual** walaupun dataset tidak punya model score.
4. Research summary tidak lagi memakai `if series:` sehingga menghindari error:
   `ValueError: The truth value of a Series is ambiguous`.

Catatan: dataset `full_labeled.parquet` adalah dataset training/continual, sehingga fokus riset ini adalah **manual signature edge terhadap label/fwd return**, bukan evaluasi trained model score kecuali score columns memang tersedia.


In [2]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    display = print

ROOT = Path.cwd()

# Default yang benar untuk riset continual.
DATASET_PATH = ROOT / "data/datasets/continual/q3_2024_to_2026_04_30/full_labeled.parquet"

# Fallback hanya untuk environment notebook/chat jika file project tidak ada.
FALLBACK_SAMPLE_CSV = Path("10-sample-full-labeled-training.csv")

OUTPUT_DIR = ROOT / "research_outputs/momentum_multistrategy_edge_v1_1_continual_safe"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_MUTUAL_INFORMATION = True
MIN_TARGET_NON_NULL = 100
TOP_K_LIST = [1, 2, 3, 5, 7, 10, 15, 20]
RANK_ASCENDING = False  # higher score = better

print("ROOT:", ROOT)
print("DATASET_PATH:", DATASET_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


ROOT: /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research
DATASET_PATH: /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/data/datasets/continual/q3_2024_to_2026_04_30/full_labeled.parquet
OUTPUT_DIR: /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/research_outputs/momentum_multistrategy_edge_v1_1_continual_safe


In [3]:
def load_dataset(path: Path) -> pd.DataFrame:
    if path.exists():
        if path.suffix.lower() == ".parquet":
            return pd.read_parquet(path)
        return pd.read_csv(path)

    if FALLBACK_SAMPLE_CSV.exists():
        print(f"WARNING: {path} not found. Using fallback sample: {FALLBACK_SAMPLE_CSV}")
        return pd.read_csv(FALLBACK_SAMPLE_CSV)

    raise FileNotFoundError(f"Dataset not found: {path}")

raw = load_dataset(DATASET_PATH)
df = raw.copy()

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
if "ticker" in df.columns:
    df["ticker"] = df["ticker"].astype(str)

audit = {
    "rows": int(len(df)),
    "cols": int(df.shape[1]),
    "min_date": str(df["date"].min().date()) if "date" in df.columns and df["date"].notna().any() else None,
    "max_date": str(df["date"].max().date()) if "date" in df.columns and df["date"].notna().any() else None,
    "unique_dates": int(df["date"].nunique()) if "date" in df.columns else None,
    "unique_tickers": int(df["ticker"].nunique()) if "ticker" in df.columns else None,
    "n_score_cols": int(sum(c.startswith("score_") for c in df.columns)),
    "n_label_cols": int(sum(c.startswith("label_") for c in df.columns)),
    "n_fwd_ret_cols": int(sum(c.startswith("fwd_ret_") for c in df.columns)),
}
with open(OUTPUT_DIR / "dataset_audit.json", "w") as f:
    json.dump(audit, f, indent=2, default=str)

audit


{'rows': 416225,
 'cols': 208,
 'min_date': '2024-07-01',
 'max_date': '2026-04-29',
 'unique_dates': 437,
 'unique_tickers': 974,
 'n_score_cols': 0,
 'n_label_cols': 10,
 'n_fwd_ret_cols': 0}

## 1. Robust target configuration

Cell ini adalah bagian yang memperbaiki masalah `TARGET_CONFIG = {}`.

Notebook tidak lagi mengasumsikan semua target/fwd return tersedia. Target akan dimasukkan jika label column ada dan punya cukup non-null row. Forward return column bersifat optional.


In [4]:
CANDIDATE_TARGETS = {
    "momentum_5d": {
        "family": "momentum_ranker",
        "label_col": "label_momentum_5d",
        "fwd_col": "fwd_ret_5d",
        "signature_col": "momentum_5d_signature_v1",
        "horizon_days": 5,
    },
    "momentum_10d": {
        "family": "momentum_ranker",
        "label_col": "label_momentum_10d",
        "fwd_col": "fwd_ret_10d",
        "signature_col": "momentum_10d_signature_v1",
        "horizon_days": 10,
    },
    "momentum_20d": {
        "family": "momentum_ranker",
        "label_col": "label_momentum_20d",
        "fwd_col": "fwd_ret_20d",
        "signature_col": "momentum_20d_signature_v1",
        "horizon_days": 20,
    },
    "scalp": {
        "family": "multi_strategy_time",
        "label_col": "label_scalp",
        "fwd_col": "fwd_ret_3d",
        "signature_col": "scalp_signature_v1",
        "horizon_days": 3,
    },
    "swing": {
        "family": "multi_strategy_time",
        "label_col": "label_swing",
        "fwd_col": "fwd_ret_5d",
        "signature_col": "swing_signature_v1",
        "horizon_days": 5,
    },
    "position": {
        "family": "multi_strategy_time",
        "label_col": "label_position",
        "fwd_col": "fwd_ret_20d",
        "signature_col": "position_signature_v1",
        "horizon_days": 20,
    },
}

def build_target_config(data: pd.DataFrame, min_non_null: int = MIN_TARGET_NON_NULL) -> dict:
    cfg = {}
    rows = []

    for target_name, spec in CANDIDATE_TARGETS.items():
        label_col = spec["label_col"]
        fwd_col = spec.get("fwd_col")

        if label_col not in data.columns:
            rows.append({
                "target": target_name,
                "label_col": label_col,
                "included": False,
                "reason": "missing_label_col",
                "label_non_null": 0,
                "label_positive_rate": None,
                "fwd_col": fwd_col,
                "fwd_col_exists": fwd_col in data.columns if fwd_col else False,
            })
            continue

        label = pd.to_numeric(data[label_col], errors="coerce")
        n_non_null = int(label.notna().sum())
        pos_rate = float(label.mean()) if n_non_null > 0 else None

        if n_non_null < min_non_null:
            rows.append({
                "target": target_name,
                "label_col": label_col,
                "included": False,
                "reason": f"label_non_null_below_{min_non_null}",
                "label_non_null": n_non_null,
                "label_positive_rate": pos_rate,
                "fwd_col": fwd_col,
                "fwd_col_exists": fwd_col in data.columns if fwd_col else False,
            })
            continue

        spec2 = dict(spec)
        if fwd_col not in data.columns:
            spec2["fwd_col"] = None

        cfg[target_name] = spec2
        rows.append({
            "target": target_name,
            "label_col": label_col,
            "included": True,
            "reason": "ok",
            "label_non_null": n_non_null,
            "label_positive_rate": pos_rate,
            "fwd_col": fwd_col,
            "fwd_col_exists": fwd_col in data.columns if fwd_col else False,
        })

    audit_df = pd.DataFrame(rows)
    audit_df.to_csv(OUTPUT_DIR / "target_config_audit_v1_1.csv", index=False)
    return cfg, audit_df

TARGET_CONFIG, target_config_audit = build_target_config(df)
print(json.dumps(TARGET_CONFIG, indent=2, default=str))
display(target_config_audit)

if not TARGET_CONFIG:
    print("WARNING: TARGET_CONFIG is empty. Check label columns in the dataset.")


{
  "momentum_5d": {
    "family": "momentum_ranker",
    "label_col": "label_momentum_5d",
    "fwd_col": null,
    "signature_col": "momentum_5d_signature_v1",
    "horizon_days": 5
  },
  "momentum_10d": {
    "family": "momentum_ranker",
    "label_col": "label_momentum_10d",
    "fwd_col": null,
    "signature_col": "momentum_10d_signature_v1",
    "horizon_days": 10
  },
  "momentum_20d": {
    "family": "momentum_ranker",
    "label_col": "label_momentum_20d",
    "fwd_col": null,
    "signature_col": "momentum_20d_signature_v1",
    "horizon_days": 20
  },
  "scalp": {
    "family": "multi_strategy_time",
    "label_col": "label_scalp",
    "fwd_col": null,
    "signature_col": "scalp_signature_v1",
    "horizon_days": 3
  },
  "swing": {
    "family": "multi_strategy_time",
    "label_col": "label_swing",
    "fwd_col": null,
    "signature_col": "swing_signature_v1",
    "horizon_days": 5
  },
  "position": {
    "family": "multi_strategy_time",
    "label_col": "label_positi

,target,label_col,included,reason,label_non_null,label_positive_rate,fwd_col,fwd_col_exists
0,momentum_5d,label_momentum_5d,True,ok,412329,0.194529,fwd_ret_5d,False
1,momentum_10d,label_momentum_10d,True,ok,407459,0.194496,fwd_ret_10d,False
2,momentum_20d,label_momentum_20d,True,ok,397723,0.196599,fwd_ret_20d,False
3,scalp,label_scalp,True,ok,416225,0.194477,fwd_ret_3d,False
4,swing,label_swing,True,ok,412329,0.194529,fwd_ret_5d,False
5,position,label_position,True,ok,397723,0.196599,fwd_ret_20d,False


In [5]:
# Target distribution
target_rows = []
for target_name, cfg in TARGET_CONFIG.items():
    label_col = cfg["label_col"]
    label = pd.to_numeric(df[label_col], errors="coerce")
    valid = label.notna()
    target_rows.append({
        "target": target_name,
        "family": cfg["family"],
        "label_col": label_col,
        "n_valid": int(valid.sum()),
        "n_positive": int((label[valid] == 1).sum()),
        "n_negative": int((label[valid] == 0).sum()),
        "positive_rate": float(label[valid].mean()) if valid.sum() else np.nan,
        "fwd_col": cfg.get("fwd_col"),
        "fwd_non_null": int(pd.to_numeric(df[cfg["fwd_col"]], errors="coerce").notna().sum()) if cfg.get("fwd_col") else 0,
    })

target_distribution = pd.DataFrame(target_rows)
target_distribution.to_csv(OUTPUT_DIR / "target_distribution_v1_1.csv", index=False)
display(target_distribution)


,target,family,label_col,n_valid,n_positive,n_negative,positive_rate,fwd_col,fwd_non_null
0,momentum_5d,momentum_ranker,label_momentum_5d,412329,80210,332119,0.194529,None,0
1,momentum_10d,momentum_ranker,label_momentum_10d,407459,79249,328210,0.194496,None,0
2,momentum_20d,momentum_ranker,label_momentum_20d,397723,78192,319531,0.196599,None,0
3,scalp,multi_strategy_time,label_scalp,416225,80946,335279,0.194477,None,0
4,swing,multi_strategy_time,label_swing,412329,80210,332119,0.194529,None,0
5,position,multi_strategy_time,label_position,397723,78192,319531,0.196599,None,0


## 2. Signature score engineering

Signature score dibuat dari cross-sectional percentile rank per tanggal. Jika suatu feature tidak ada di dataset, feature tersebut otomatis di-skip.


In [6]:
def add_cs_rank(data: pd.DataFrame, col: str, higher_is_better: bool = True) -> pd.Series:
    x = pd.to_numeric(data[col], errors="coerce")
    if "date" in data.columns:
        # pct=True menghasilkan 0..1. higher score should be higher rank.
        if higher_is_better:
            return x.groupby(data["date"]).rank(pct=True, ascending=True)
        return x.groupby(data["date"]).rank(pct=True, ascending=False)
    return x.rank(pct=True, ascending=True if higher_is_better else False)

def weighted_rank_score(data: pd.DataFrame, components: list[tuple[str, float, bool]]) -> pd.Series:
    total = pd.Series(0.0, index=data.index)
    used_weight = pd.Series(0.0, index=data.index)

    for col, weight, higher_is_better in components:
        if col not in data.columns:
            continue
        r = add_cs_rank(data, col, higher_is_better=higher_is_better)
        total = total.add(r.fillna(0.0) * weight, fill_value=0.0)
        used_weight = used_weight.add((~r.isna()).astype(float) * abs(weight), fill_value=0.0)

    out = total / used_weight.replace(0, np.nan)
    return out

SIGNATURE_COMPONENTS = {
    "momentum_5d_signature_v1": [
        ("ret_1d", 0.12, True),
        ("ret_5d", 0.24, True),
        ("ret_10d", 0.14, True),
        ("volume_ratio_20d", 0.14, True),
        ("close_vs_ma20", 0.14, True),
        ("volatility_20d", 0.08, True),
        ("traded_value_proxy", 0.08, True),
        ("buyer_dominance_ratio", 0.04, True),
        ("net_flow_ratio", 0.02, True),
    ],
    "momentum_10d_signature_v1": [
        ("ret_5d", 0.16, True),
        ("ret_10d", 0.24, True),
        ("ret_20d", 0.14, True),
        ("volume_ratio_20d", 0.10, True),
        ("close_vs_ma20", 0.16, True),
        ("volatility_20d", 0.08, True),
        ("traded_value_proxy", 0.08, True),
        ("buyer_dominance_ratio", 0.02, True),
        ("net_flow_ratio", 0.02, True),
    ],
    "momentum_20d_signature_v1": [
        ("ret_10d", 0.18, True),
        ("ret_20d", 0.26, True),
        ("close_vs_ma20", 0.18, True),
        ("volume_ratio_20d", 0.08, True),
        ("volatility_20d", 0.08, True),
        ("traded_value_proxy", 0.12, True),
        ("buyer_dominance_ratio", 0.04, True),
        ("net_flow_ratio", 0.04, True),
        ("rank1_same_buyer_streak", 0.02, True),
    ],
    "scalp_signature_v1": [
        ("ret_1d", 0.18, True),
        ("volume_ratio_20d", 0.18, True),
        ("volatility_20d", 0.14, True),
        ("traded_value_proxy", 0.14, True),
        ("buyer_dominance_ratio", 0.14, True),
        ("net_flow_ratio", 0.12, True),
        ("rank1_same_buyer_streak", 0.10, True),
    ],
    "swing_signature_v1": [
        ("ret_1d", 0.08, True),
        ("ret_5d", 0.20, True),
        ("ret_10d", 0.16, True),
        ("volume_ratio_20d", 0.12, True),
        ("close_vs_ma20", 0.18, True),
        ("volatility_20d", 0.08, True),
        ("traded_value_proxy", 0.10, True),
        ("buyer_dominance_ratio", 0.04, True),
        ("net_flow_ratio", 0.04, True),
    ],
    "position_signature_v1": [
        ("ret_10d", 0.14, True),
        ("ret_20d", 0.26, True),
        ("close_vs_ma20", 0.20, True),
        ("traded_value_proxy", 0.16, True),
        ("volume_ratio_20d", 0.08, True),
        ("volatility_20d", 0.06, True),
        ("buyer_dominance_ratio", 0.04, True),
        ("net_flow_ratio", 0.04, True),
        ("rank1_same_buyer_streak", 0.02, True),
    ],
}

work = df.copy()
signature_audit = []
for sig_col, comps in SIGNATURE_COMPONENTS.items():
    work[sig_col] = weighted_rank_score(work, comps)
    used_cols = [col for col, _, _ in comps if col in work.columns]
    signature_audit.append({
        "signature_col": sig_col,
        "used_cols": ",".join(used_cols),
        "n_used_cols": len(used_cols),
        "non_null": int(work[sig_col].notna().sum()),
        "min": float(work[sig_col].min()) if work[sig_col].notna().any() else np.nan,
        "max": float(work[sig_col].max()) if work[sig_col].notna().any() else np.nan,
    })

signature_audit_df = pd.DataFrame(signature_audit)
signature_audit_df.to_csv(OUTPUT_DIR / "signature_audit_v1_1.csv", index=False)
display(signature_audit_df)


,signature_col,used_cols,n_used_cols,non_null,min,max
0,momentum_5d_signature_v1,"ret_1d,ret_5d,ret_10d,volume_ratio_20d,close_v...",9,416225,0.048925,0.994715
1,momentum_10d_signature_v1,"ret_5d,ret_10d,ret_20d,volume_ratio_20d,close_...",9,416225,0.011384,0.994715
2,momentum_20d_signature_v1,"ret_10d,ret_20d,close_vs_ma20,volume_ratio_20d...",9,416225,0.022829,0.989648
3,scalp_signature_v1,"ret_1d,volume_ratio_20d,volatility_20d,traded_...",7,416225,0.048925,0.984724
4,swing_signature_v1,"ret_1d,ret_5d,ret_10d,volume_ratio_20d,close_v...",9,416225,0.048925,0.994715
5,position_signature_v1,"ret_10d,ret_20d,close_vs_ma20,traded_value_pro...",9,416225,0.030089,0.988096


## 3. Top-k edge evaluation — safe mode

Cell ini memperbaiki error:

`KeyError: 'target'`

Penyebab error sebelumnya: `eval_rows` kosong lalu `pd.DataFrame(eval_rows).sort_values(["target", ...])`.

Versi ini:
- selalu mengevaluasi `signature_col`,
- mengevaluasi `score_*` hanya jika tersedia,
- tidak error ketika rows kosong,
- tetap menghasilkan file CSV dengan schema kosong.


In [7]:
TOPK_EVAL_COLUMNS = [
    "target", "family", "ranker_type", "score_col", "top_k", "n_candidates", "n_days",
    "label_col", "label_rate", "baseline_label_rate", "label_lift_vs_baseline",
    "fwd_col", "avg_forward_return", "baseline_avg_forward_return", "avg_forward_return_spread",
    "median_forward_return", "positive_return_rate", "baseline_positive_return_rate",
]

def evaluate_ranker_topk(data: pd.DataFrame, target_name: str, cfg: dict, score_col: str, ranker_type: str) -> list[dict]:
    label_col = cfg["label_col"]
    fwd_col = cfg.get("fwd_col")

    if score_col not in data.columns or label_col not in data.columns:
        return []

    sub = data.copy()
    sub["_label"] = pd.to_numeric(sub[label_col], errors="coerce")
    sub["_score"] = pd.to_numeric(sub[score_col], errors="coerce")

    valid_cols = sub["_label"].notna() & sub["_score"].notna()
    if "date" in sub.columns:
        valid_cols &= sub["date"].notna()
    sub = sub.loc[valid_cols].copy()

    if sub.empty:
        return []

    if fwd_col and fwd_col in sub.columns:
        sub["_fwd"] = pd.to_numeric(sub[fwd_col], errors="coerce")
    else:
        sub["_fwd"] = np.nan
        fwd_col = None

    baseline_label_rate = float(sub["_label"].mean()) if len(sub) else np.nan
    baseline_avg_fwd = float(sub["_fwd"].mean()) if fwd_col and sub["_fwd"].notna().any() else np.nan
    baseline_pos_rate = float((sub["_fwd"] > 0).mean()) if fwd_col and sub["_fwd"].notna().any() else np.nan

    if "date" in sub.columns:
        sub["_rank"] = sub.groupby("date")["_score"].rank(method="first", ascending=RANK_ASCENDING)
        n_days_all = int(sub["date"].nunique())
    else:
        sub["_rank"] = sub["_score"].rank(method="first", ascending=RANK_ASCENDING)
        n_days_all = 1

    rows = []
    for k in TOP_K_LIST:
        top = sub[sub["_rank"] <= k].copy()
        if top.empty:
            continue

        avg_fwd = float(top["_fwd"].mean()) if fwd_col and top["_fwd"].notna().any() else np.nan
        med_fwd = float(top["_fwd"].median()) if fwd_col and top["_fwd"].notna().any() else np.nan
        pos_rate = float((top["_fwd"] > 0).mean()) if fwd_col and top["_fwd"].notna().any() else np.nan
        label_rate = float(top["_label"].mean()) if top["_label"].notna().any() else np.nan

        rows.append({
            "target": target_name,
            "family": cfg["family"],
            "ranker_type": ranker_type,
            "score_col": score_col,
            "top_k": int(k),
            "n_candidates": int(len(top)),
            "n_days": int(top["date"].nunique()) if "date" in top.columns else n_days_all,
            "label_col": label_col,
            "label_rate": label_rate,
            "baseline_label_rate": baseline_label_rate,
            "label_lift_vs_baseline": float(label_rate / baseline_label_rate) if baseline_label_rate and baseline_label_rate > 0 else np.nan,
            "fwd_col": fwd_col,
            "avg_forward_return": avg_fwd,
            "baseline_avg_forward_return": baseline_avg_fwd,
            "avg_forward_return_spread": float(avg_fwd - baseline_avg_fwd) if fwd_col and pd.notna(avg_fwd) and pd.notna(baseline_avg_fwd) else np.nan,
            "median_forward_return": med_fwd,
            "positive_return_rate": pos_rate,
            "baseline_positive_return_rate": baseline_pos_rate,
        })

    return rows

score_cols = [c for c in work.columns if c.startswith("score_")]
eval_rows = []

for target_name, cfg in TARGET_CONFIG.items():
    # Always evaluate manual signature first.
    sig_col = cfg.get("signature_col")
    if sig_col and sig_col in work.columns:
        eval_rows.extend(evaluate_ranker_topk(work, target_name, cfg, sig_col, "manual_signature"))

    # Optional: evaluate model score columns if dataset has any.
    for sc in score_cols:
        compact_target = target_name.replace("_", "")
        compact_score = sc.replace("_", "")
        if compact_target in compact_score or cfg["family"] in sc:
            eval_rows.extend(evaluate_ranker_topk(work, target_name, cfg, sc, "model_score"))

if eval_rows:
    topk_eval = pd.DataFrame(eval_rows)
    topk_eval = topk_eval.sort_values(["target", "ranker_type", "score_col", "top_k"]).reset_index(drop=True)
else:
    topk_eval = pd.DataFrame(columns=TOPK_EVAL_COLUMNS)

topk_eval.to_csv(OUTPUT_DIR / "signature_topk_edge_evaluation_v1_1.csv", index=False)
display(topk_eval.head(100))
print("Rows:", len(topk_eval))


,target,family,ranker_type,score_col,top_k,n_candidates,n_days,label_col,label_rate,baseline_label_rate,label_lift_vs_baseline,fwd_col,avg_forward_return,baseline_avg_forward_return,avg_forward_return_spread,median_forward_return,positive_return_rate,baseline_positive_return_rate
0,momentum_10d,momentum_ranker,manual_signature,momentum_10d_signature_v1,1,428,428,label_momentum_10d,0.331776,0.194496,1.705826,None,NaN,NaN,NaN,NaN,NaN,NaN
1,momentum_10d,momentum_ranker,manual_signature,momentum_10d_signature_v1,2,856,428,label_momentum_10d,0.344626,0.194496,1.771897,None,NaN,NaN,NaN,NaN,NaN,NaN
2,momentum_10d,momentum_ranker,manual_signature,momentum_10d_signature_v1,3,1284,428,label_momentum_10d,0.351246,0.194496,1.805933,None,NaN,NaN,NaN,NaN,NaN,NaN
3,momentum_10d,momentum_ranker,manual_signature,momentum_10d_signature_v1,5,2140,428,label_momentum_10d,0.350935,0.194496,1.804331,None,NaN,NaN,NaN,NaN,NaN,NaN
4,momentum_10d,momentum_ranker,manual_signature,momentum_10d_signature_v1,7,2996,428,label_momentum_10d,0.350467,0.194496,1.801929,None,NaN,NaN,NaN,NaN,NaN,NaN
5,momentum_10d,momentum_ranker,manual_signature,momentum_10d_signature_v1,10,4280,428,label_momentum_10d,0.350701,0.194496,1.803130,None,NaN,NaN,NaN,NaN,NaN,NaN
6,momentum_10d,momentum_ranker,manual_signature,momentum_10d_signature_v1,15,6420,428,label_momentum_10d,0.342679,0.194496,1.761886,None,NaN,NaN,NaN,NaN,NaN,NaN
7,momentum_10d,momentum_ranker,manual_signature,momentum_10d_signature_v1,20,8560,428,label_momentum_10d,0.339603,0.194496,1.746069,None,NaN,NaN,NaN,NaN,NaN,NaN
8,momentum_20d,momentum_ranker,manual_signature,momentum_20d_signature_v1,1,418,418,label_momentum_20d,0.315789,0.196599,1.606261,None,NaN,NaN,NaN,NaN,NaN,NaN
9,momentum_20d,momentum_ranker,manual_signature,momentum_20d_signature_v1,2,836,418,label_momentum_20d,0.300239,0.196599,1.527165,None,NaN,NaN,NaN,NaN,NaN,NaN


Rows: 48


## 4. Event capture by signature

Menampilkan event positive yang masuk top-k signature. Aman walaupun target kosong.


In [8]:
event_rows = []

for target_name, cfg in TARGET_CONFIG.items():
    label_col = cfg["label_col"]
    sig_col = cfg.get("signature_col")
    fwd_col = cfg.get("fwd_col")

    if not sig_col or sig_col not in work.columns or label_col not in work.columns:
        continue

    sub = work.copy()
    sub["_label"] = pd.to_numeric(sub[label_col], errors="coerce")
    sub["_score"] = pd.to_numeric(sub[sig_col], errors="coerce")
    if fwd_col and fwd_col in sub.columns:
        sub["_fwd"] = pd.to_numeric(sub[fwd_col], errors="coerce")
    else:
        sub["_fwd"] = np.nan

    sub = sub[sub["_label"].notna() & sub["_score"].notna()].copy()
    if sub.empty:
        continue

    if "date" in sub.columns:
        sub["_rank"] = sub.groupby("date")["_score"].rank(method="first", ascending=False)
    else:
        sub["_rank"] = sub["_score"].rank(method="first", ascending=False)

    ev = sub[sub["_label"] == 1].copy()
    if ev.empty:
        continue

    cols = ["date", "ticker"] if "date" in ev.columns and "ticker" in ev.columns else []
    for _, r in ev.sort_values("_rank").head(200).iterrows():
        row = {
            "target": target_name,
            "signature_col": sig_col,
            "label_col": label_col,
            "date": r.get("date", pd.NaT),
            "ticker": r.get("ticker", None),
            "rank": int(r["_rank"]) if pd.notna(r["_rank"]) else None,
            "signature_score": float(r["_score"]) if pd.notna(r["_score"]) else np.nan,
            "fwd_col": fwd_col,
            "forward_return": float(r["_fwd"]) if pd.notna(r["_fwd"]) else np.nan,
        }
        event_rows.append(row)

event_capture = pd.DataFrame(event_rows)
if not event_capture.empty:
    event_capture = event_capture.sort_values(["target", "rank"]).reset_index(drop=True)
event_capture.to_csv(OUTPUT_DIR / "event_capture_by_signature_v1_1.csv", index=False)
display(event_capture.head(100))
print("Rows:", len(event_capture))


,target,signature_col,label_col,date,ticker,rank,signature_score,fwd_col,forward_return
0,momentum_10d,momentum_10d_signature_v1,label_momentum_10d,2025-11-03,UVCR,1,0.959364,None,NaN
1,momentum_10d,momentum_10d_signature_v1,label_momentum_10d,2024-08-13,MSIN,1,0.949098,None,NaN
2,momentum_10d,momentum_10d_signature_v1,label_momentum_10d,2024-10-31,GPSO,1,0.965290,None,NaN
3,momentum_10d,momentum_10d_signature_v1,label_momentum_10d,2025-09-30,KOKA,1,0.964207,None,NaN
4,momentum_10d,momentum_10d_signature_v1,label_momentum_10d,2024-10-18,MLPL,1,0.975914,None,NaN
...,...,...,...,...,...,...,...,...,...
95,momentum_10d,momentum_10d_signature_v1,label_momentum_10d,2025-06-26,PTMP,1,0.975779,None,NaN
96,momentum_10d,momentum_10d_signature_v1,label_momentum_10d,2025-06-18,KRAS,1,0.967856,None,NaN
97,momentum_10d,momentum_10d_signature_v1,label_momentum_10d,2024-08-20,LABA,1,0.951342,None,NaN
98,momentum_10d,momentum_10d_signature_v1,label_momentum_10d,2024-08-22,LABA,1,0.969131,None,NaN


Rows: 1200


## 5. Feature distribution: positive vs non-positive label

Tujuan: melihat feature apa yang berbeda antara label positive dan non-positive per target.


In [9]:
EXCLUDE_PREFIXES = ("label_", "target_", "score_")
EXCLUDE_COLS = {"date", "ticker", "rank1_buyer", "rank1_seller", "rank1_buyer_type", "rank1_seller_type", "market_regime"}

candidate_numeric_features = []
for c in work.columns:
    if c in EXCLUDE_COLS:
        continue
    if any(c.startswith(p) for p in EXCLUDE_PREFIXES):
        continue
    if c.startswith("fwd_ret_"):
        continue
    if pd.api.types.is_numeric_dtype(work[c]):
        candidate_numeric_features.append(c)

feature_dist_rows = []
for target_name, cfg in TARGET_CONFIG.items():
    label_col = cfg["label_col"]
    if label_col not in work.columns:
        continue
    label = pd.to_numeric(work[label_col], errors="coerce")
    valid = label.notna()
    pos_mask = valid & (label == 1)
    neg_mask = valid & (label == 0)

    for feat in candidate_numeric_features:
        x = pd.to_numeric(work[feat], errors="coerce")
        if x[valid].notna().sum() < 100:
            continue

        pos_med = x[pos_mask].median()
        neg_med = x[neg_mask].median()
        pos_mean = x[pos_mask].mean()
        neg_mean = x[neg_mask].mean()

        feature_dist_rows.append({
            "target": target_name,
            "feature": feat,
            "n_non_null": int(x[valid].notna().sum()),
            "positive_median": float(pos_med) if pd.notna(pos_med) else np.nan,
            "non_positive_median": float(neg_med) if pd.notna(neg_med) else np.nan,
            "median_diff": float(pos_med - neg_med) if pd.notna(pos_med) and pd.notna(neg_med) else np.nan,
            "positive_mean": float(pos_mean) if pd.notna(pos_mean) else np.nan,
            "non_positive_mean": float(neg_mean) if pd.notna(neg_mean) else np.nan,
            "mean_diff": float(pos_mean - neg_mean) if pd.notna(pos_mean) and pd.notna(neg_mean) else np.nan,
        })

feature_dist = pd.DataFrame(feature_dist_rows)
if not feature_dist.empty:
    feature_dist = feature_dist.sort_values(["target", "median_diff"], ascending=[True, False])
feature_dist.to_csv(OUTPUT_DIR / "feature_distribution_positive_vs_non_positive_v1_1.csv", index=False)
display(feature_dist.head(80))


,target,feature,n_non_null,positive_median,non_positive_median,median_diff,positive_mean,non_positive_mean,mean_diff
201,momentum_10d,value,14375,7.513973e+08,2.534657e+08,4.979316e+08,1.789833e+10,1.624072e+10,1.657608e+09
215,momentum_10d,traded_value_proxy,407459,3.613050e+08,1.827175e+08,1.785875e+08,1.841738e+10,1.492221e+10,3.495164e+09
223,momentum_10d,buy_val_total_sane,360474,4.455153e+08,3.762062e+08,6.930910e+07,2.099866e+10,1.918027e+10,1.818391e+09
224,momentum_10d,sell_val_total_sane,360473,4.455476e+08,3.762409e+08,6.930675e+07,2.102243e+10,1.917142e+10,1.851013e+09
217,momentum_10d,buy_val_total,360486,4.455305e+08,3.762379e+08,6.929260e+07,2.160835e+10,2.021393e+10,1.394427e+09
...,...,...,...,...,...,...,...,...,...
390,momentum_10d,swing_signature_v1,407459,5.097116e-01,4.677775e-01,4.193410e-02,5.267397e-01,4.922600e-01,3.447974e-02
386,momentum_10d,momentum_5d_signature_v1,407459,5.052766e-01,4.691930e-01,3.608363e-02,5.255617e-01,4.928494e-01,3.271228e-02
274,momentum_10d,ihsg_zscore,407459,7.271444e-01,6.962806e-01,3.086379e-02,2.887441e-01,2.788276e-01,9.916438e-03
277,momentum_10d,wti_brent_spread,407459,-3.649998e+00,-3.669998e+00,2.000046e-02,-3.574177e+00,-3.614796e+00,4.061840e-02


## 6. Probabilistic threshold discovery

Mencari feature threshold percentile yang menaikkan label rate dan return spread.


In [10]:
THRESHOLD_QUANTILES = [0.50, 0.60, 0.70, 0.75, 0.80, 0.90, 0.95]
threshold_rows = []

for target_name, cfg in TARGET_CONFIG.items():
    label_col = cfg["label_col"]
    fwd_col = cfg.get("fwd_col")
    label = pd.to_numeric(work[label_col], errors="coerce")
    valid_label = label.notna()
    baseline_rate = label[valid_label].mean() if valid_label.any() else np.nan

    fwd = pd.to_numeric(work[fwd_col], errors="coerce") if fwd_col and fwd_col in work.columns else pd.Series(np.nan, index=work.index)
    baseline_fwd = fwd[valid_label].mean() if fwd.notna().any() else np.nan

    for feat in candidate_numeric_features:
        x = pd.to_numeric(work[feat], errors="coerce")
        valid = valid_label & x.notna()
        if valid.sum() < 500:
            continue

        for q in THRESHOLD_QUANTILES:
            thr = x[valid].quantile(q)
            mask = valid & (x >= thr)
            support = int(mask.sum())
            if support < 30:
                continue

            rate = label[mask].mean()
            avg_fwd = fwd[mask].mean() if fwd.notna().any() else np.nan

            threshold_rows.append({
                "target": target_name,
                "feature": feat,
                "quantile": q,
                "threshold": float(thr) if pd.notna(thr) else np.nan,
                "support": support,
                "label_rate": float(rate) if pd.notna(rate) else np.nan,
                "baseline_label_rate": float(baseline_rate) if pd.notna(baseline_rate) else np.nan,
                "label_lift": float(rate / baseline_rate) if baseline_rate and baseline_rate > 0 and pd.notna(rate) else np.nan,
                "avg_forward_return": float(avg_fwd) if pd.notna(avg_fwd) else np.nan,
                "baseline_avg_forward_return": float(baseline_fwd) if pd.notna(baseline_fwd) else np.nan,
                "avg_forward_return_spread": float(avg_fwd - baseline_fwd) if pd.notna(avg_fwd) and pd.notna(baseline_fwd) else np.nan,
            })

threshold_df = pd.DataFrame(threshold_rows)
if not threshold_df.empty:
    threshold_df = threshold_df.sort_values(
        ["target", "label_lift", "avg_forward_return_spread", "support"],
        ascending=[True, False, False, False],
    )
threshold_df.to_csv(OUTPUT_DIR / "probabilistic_threshold_discovery_momentum_multistrategy_v1_1.csv", index=False)
display(threshold_df.head(100))


,target,feature,quantile,threshold,support,label_rate,baseline_label_rate,label_lift,avg_forward_return,baseline_avg_forward_return,avg_forward_return_spread
2297,momentum_10d,bdm_non_retail_day_hist2,0.90,60688.519600,56,0.375000,0.194496,1.928064,NaN,NaN,NaN
1391,momentum_10d,close_vs_ma20,0.95,0.204819,19463,0.370292,0.194496,1.903859,NaN,NaN,NaN
2251,momentum_10d,bdm_market_maker_week_hist3,0.90,42333.089400,76,0.368421,0.194496,1.894238,NaN,NaN,NaN
1356,momentum_10d,ret_20d,0.95,0.447950,19400,0.364433,0.194496,1.873733,NaN,NaN,NaN
2243,momentum_10d,bdm_market_maker_week_hist2,0.80,19020.725000,151,0.364238,0.194496,1.872733,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2306,momentum_10d,bdm_non_retail_day_hist4,0.70,4392.943800,167,0.293413,0.194496,1.508585,NaN,NaN,NaN
1389,momentum_10d,close_vs_ma20,0.80,0.041850,77794,0.292748,0.194496,1.505162,NaN,NaN,NaN
2490,momentum_10d,scalp_signature_v1,0.95,0.721863,20373,0.292397,0.194496,1.503359,NaN,NaN,NaN
2255,momentum_10d,bdm_market_maker_week_hist4,0.70,6636.817300,226,0.292035,0.194496,1.501501,NaN,NaN,NaN


## 7. Simple feature interaction discovery

Menguji pair feature high-percentile sederhana. Ini research-only, bukan production rule.


In [11]:
INTERACTION_FEATURE_POOL = [
    "ret_1d", "ret_5d", "ret_10d", "ret_20d",
    "volume_ratio_20d", "close_vs_ma20", "volatility_20d",
    "traded_value_proxy", "buyer_dominance_ratio", "net_flow_ratio",
    "rank1_same_buyer_streak",
]
INTERACTION_FEATURE_POOL = [c for c in INTERACTION_FEATURE_POOL if c in work.columns]
INTERACTION_Q = 0.75

interaction_rows = []
for target_name, cfg in TARGET_CONFIG.items():
    label_col = cfg["label_col"]
    fwd_col = cfg.get("fwd_col")
    label = pd.to_numeric(work[label_col], errors="coerce")
    valid_label = label.notna()
    baseline_rate = label[valid_label].mean() if valid_label.any() else np.nan
    fwd = pd.to_numeric(work[fwd_col], errors="coerce") if fwd_col and fwd_col in work.columns else pd.Series(np.nan, index=work.index)
    baseline_fwd = fwd[valid_label].mean() if fwd.notna().any() else np.nan

    thresholds = {}
    for feat in INTERACTION_FEATURE_POOL:
        x = pd.to_numeric(work[feat], errors="coerce")
        valid = valid_label & x.notna()
        if valid.sum() >= 500:
            thresholds[feat] = x[valid].quantile(INTERACTION_Q)

    feats = list(thresholds.keys())
    for i in range(len(feats)):
        for j in range(i + 1, len(feats)):
            f1, f2 = feats[i], feats[j]
            x1 = pd.to_numeric(work[f1], errors="coerce")
            x2 = pd.to_numeric(work[f2], errors="coerce")
            mask = valid_label & (x1 >= thresholds[f1]) & (x2 >= thresholds[f2])
            support = int(mask.sum())
            if support < 30:
                continue
            rate = label[mask].mean()
            avg_fwd = fwd[mask].mean() if fwd.notna().any() else np.nan
            interaction_rows.append({
                "target": target_name,
                "feature_1": f1,
                "feature_2": f2,
                "quantile_rule": f">= p{int(INTERACTION_Q*100)} both",
                "support": support,
                "label_rate": float(rate) if pd.notna(rate) else np.nan,
                "baseline_label_rate": float(baseline_rate) if pd.notna(baseline_rate) else np.nan,
                "label_lift": float(rate / baseline_rate) if baseline_rate and baseline_rate > 0 and pd.notna(rate) else np.nan,
                "avg_forward_return": float(avg_fwd) if pd.notna(avg_fwd) else np.nan,
                "baseline_avg_forward_return": float(baseline_fwd) if pd.notna(baseline_fwd) else np.nan,
                "avg_forward_return_spread": float(avg_fwd - baseline_fwd) if pd.notna(avg_fwd) and pd.notna(baseline_fwd) else np.nan,
            })

interaction_df = pd.DataFrame(interaction_rows)
if not interaction_df.empty:
    interaction_df = interaction_df.sort_values(
        ["target", "label_lift", "avg_forward_return_spread", "support"],
        ascending=[True, False, False, False],
    )
interaction_df.to_csv(OUTPUT_DIR / "simple_feature_interaction_discovery_momentum_multistrategy_v1_1.csv", index=False)
display(interaction_df.head(100))


,target,feature_1,feature_2,quantile_rule,support,label_rate,baseline_label_rate,label_lift,avg_forward_return,baseline_avg_forward_return,avg_forward_return_spread
57,momentum_10d,ret_1d,ret_20d,>= p75 both,39942,0.315833,0.194496,1.623856,NaN,NaN,NaN
86,momentum_10d,ret_20d,buyer_dominance_ratio,>= p75 both,19340,0.315202,0.194496,1.620610,NaN,NaN,NaN
101,momentum_10d,volatility_20d,buyer_dominance_ratio,>= p75 both,16927,0.312932,0.194496,1.608941,NaN,NaN,NaN
82,momentum_10d,ret_20d,volume_ratio_20d,>= p75 both,31997,0.308779,0.194496,1.587588,NaN,NaN,NaN
66,momentum_10d,ret_5d,ret_20d,>= p75 both,53111,0.308524,0.194496,1.586275,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
147,momentum_20d,volume_ratio_20d,buyer_dominance_ratio,>= p75 both,20902,0.244283,0.196599,1.242543,NaN,NaN,NaN
133,momentum_20d,ret_10d,traded_value_proxy,>= p75 both,37080,0.244067,0.196599,1.241444,NaN,NaN,NaN
128,momentum_20d,ret_5d,rank1_same_buyer_streak,>= p75 both,33060,0.241500,0.196599,1.228389,NaN,NaN,NaN
158,momentum_20d,volatility_20d,rank1_same_buyer_streak,>= p75 both,35514,0.236780,0.196599,1.204379,NaN,NaN,NaN


## 8. Mutual information scan

Optional. Dijalankan jika `RUN_MUTUAL_INFORMATION=True`. Dibuat safe agar tidak error bila sklearn tidak tersedia atau target kosong.


In [12]:
mi_df = pd.DataFrame()

if RUN_MUTUAL_INFORMATION and TARGET_CONFIG:
    try:
        from sklearn.feature_selection import mutual_info_classif
        from sklearn.impute import SimpleImputer
        from sklearn.preprocessing import RobustScaler

        mi_rows = []
        mi_features = []
        for c in candidate_numeric_features:
            x = pd.to_numeric(work[c], errors="coerce")
            if x.notna().sum() >= 1000 and x.nunique(dropna=True) > 5:
                mi_features.append(c)

        # limit for speed; prioritize common technical/liquidity/broker features, then all.
        priority = [
            "ret_1d", "ret_5d", "ret_10d", "ret_20d",
            "volume_ratio_20d", "close_vs_ma20", "volatility_20d", "traded_value_proxy",
            "buyer_dominance_ratio", "net_flow_ratio", "rank1_same_buyer_streak",
            "market_ret_5d", "market_ret_20d",
        ]
        mi_features = [c for c in priority if c in mi_features] + [c for c in mi_features if c not in priority]
        mi_features = mi_features[:80]

        for target_name, cfg in TARGET_CONFIG.items():
            label_col = cfg["label_col"]
            y = pd.to_numeric(work[label_col], errors="coerce")
            valid = y.notna()
            if valid.sum() < 1000 or y[valid].nunique() < 2:
                continue

            X = work.loc[valid, mi_features].apply(pd.to_numeric, errors="coerce")
            yv = y.loc[valid].astype(int)

            X_imp = SimpleImputer(strategy="median").fit_transform(X)
            X_scaled = RobustScaler().fit_transform(X_imp)
            mi = mutual_info_classif(X_scaled, yv, random_state=42, discrete_features=False)

            for feat, val in zip(mi_features, mi):
                mi_rows.append({
                    "target": target_name,
                    "feature": feat,
                    "mutual_information": float(val),
                    "n_rows": int(valid.sum()),
                })

        mi_df = pd.DataFrame(mi_rows)
        if not mi_df.empty:
            mi_df = mi_df.sort_values(["target", "mutual_information"], ascending=[True, False])
    except Exception as e:
        print("Mutual information skipped due to error:", repr(e))
        mi_df = pd.DataFrame()

mi_df.to_csv(OUTPUT_DIR / "feature_mutual_information_momentum_multistrategy_v1_1.csv", index=False)
display(mi_df.head(100))


,target,feature,mutual_information,n_rows
86,momentum_10d,volatility_20d,0.047505,407459
103,momentum_10d,ma_20,0.026797,407459
85,momentum_10d,close_vs_ma20,0.022903,407459
102,momentum_10d,ma_5,0.022531,407459
95,momentum_10d,low,0.022256,407459
...,...,...,...,...
192,momentum_20d,sell_val_total_sane,0.012599,397723
186,momentum_20d,sell_val_total,0.012439,397723
160,momentum_20d,ret_1d,0.012269,397723
195,momentum_20d,rank1_sell_val_sane,0.012230,397723


## 9. Research summary — safe mode

Tidak memakai `if series:` sehingga tidak memunculkan `ValueError: The truth value of a Series is ambiguous`.


In [13]:
def safe_top_records(d: pd.DataFrame, n: int = 10) -> list[dict]:
    if d is None or d.empty:
        return []
    return json.loads(d.head(n).replace({np.nan: None}).to_json(orient="records", date_format="iso"))

summary = {
    "dataset_audit": audit,
    "target_config": TARGET_CONFIG,
    "n_targets": len(TARGET_CONFIG),
    "target_distribution": safe_top_records(target_distribution, 20),
    "signature_audit": safe_top_records(signature_audit_df, 20),
    "topk_best_by_target": {},
    "threshold_best_by_target": {},
    "interaction_best_by_target": {},
    "warnings": [],
}

if not TARGET_CONFIG:
    summary["warnings"].append("TARGET_CONFIG is empty. Check label column names and min_non_null threshold.")
if topk_eval.empty:
    summary["warnings"].append("Top-k evaluation is empty. Check signature score non-null and target labels.")
if audit.get("n_score_cols", 0) == 0:
    summary["warnings"].append("No score_* columns found. Evaluation is based on manual signature scores only.")

# Best top-k per target: prioritize label lift and return spread.
if not topk_eval.empty:
    sort_cols = ["label_lift_vs_baseline"]
    asc = [False]
    if "avg_forward_return_spread" in topk_eval.columns:
        sort_cols.append("avg_forward_return_spread")
        asc.append(False)
    best = (
        topk_eval.sort_values(sort_cols, ascending=asc)
        .groupby("target", as_index=False)
        .head(5)
    )
    for target_name, g in best.groupby("target"):
        summary["topk_best_by_target"][target_name] = safe_top_records(g, 5)

if not threshold_df.empty:
    best_thr = (
        threshold_df.sort_values(["label_lift", "avg_forward_return_spread", "support"], ascending=[False, False, False])
        .groupby("target", as_index=False)
        .head(5)
    )
    for target_name, g in best_thr.groupby("target"):
        summary["threshold_best_by_target"][target_name] = safe_top_records(g, 5)

if not interaction_df.empty:
    best_inter = (
        interaction_df.sort_values(["label_lift", "avg_forward_return_spread", "support"], ascending=[False, False, False])
        .groupby("target", as_index=False)
        .head(5)
    )
    for target_name, g in best_inter.groupby("target"):
        summary["interaction_best_by_target"][target_name] = safe_top_records(g, 5)

with open(OUTPUT_DIR / "research_summary_momentum_multistrategy_v1_1.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

summary


{'dataset_audit': {'rows': 416225,
  'cols': 208,
  'min_date': '2024-07-01',
  'max_date': '2026-04-29',
  'unique_dates': 437,
  'unique_tickers': 974,
  'n_score_cols': 0,
  'n_label_cols': 10,
  'n_fwd_ret_cols': 0},
 'target_config': {'momentum_5d': {'family': 'momentum_ranker',
   'label_col': 'label_momentum_5d',
   'fwd_col': None,
   'signature_col': 'momentum_5d_signature_v1',
   'horizon_days': 5},
  'momentum_10d': {'family': 'momentum_ranker',
   'label_col': 'label_momentum_10d',
   'fwd_col': None,
   'signature_col': 'momentum_10d_signature_v1',
   'horizon_days': 10},
  'momentum_20d': {'family': 'momentum_ranker',
   'label_col': 'label_momentum_20d',
   'fwd_col': None,
   'signature_col': 'momentum_20d_signature_v1',
   'horizon_days': 20},
  'scalp': {'family': 'multi_strategy_time',
   'label_col': 'label_scalp',
   'fwd_col': None,
   'signature_col': 'scalp_signature_v1',
   'horizon_days': 3},
  'swing': {'family': 'multi_strategy_time',
   'label_col': 'label_

## 10. Export compact enriched columns

File compact agar ukuran tidak terlalu besar.


In [14]:
core_cols = ["date", "ticker"]
for target_name, cfg in TARGET_CONFIG.items():
    for c in [cfg["label_col"], cfg.get("fwd_col"), cfg.get("signature_col")]:
        if c and c in work.columns and c not in core_cols:
            core_cols.append(c)

for c in [
    "ret_1d", "ret_5d", "ret_10d", "ret_20d",
    "volume_ratio_20d", "close_vs_ma20", "volatility_20d",
    "traded_value_proxy", "buyer_dominance_ratio", "net_flow_ratio",
    "rank1_same_buyer_streak", "market_regime",
]:
    if c in work.columns and c not in core_cols:
        core_cols.append(c)

compact = work[core_cols].copy()
compact.to_csv(OUTPUT_DIR / "signature_enriched_core_columns_v1_1.csv", index=False)
print("Exported:", OUTPUT_DIR / "signature_enriched_core_columns_v1_1.csv")
print("Columns:", core_cols)
display(compact.head())


Exported: /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/research_outputs/momentum_multistrategy_edge_v1_1_continual_safe/signature_enriched_core_columns_v1_1.csv
Columns: ['date', 'ticker', 'label_momentum_5d', 'momentum_5d_signature_v1', 'label_momentum_10d', 'momentum_10d_signature_v1', 'label_momentum_20d', 'momentum_20d_signature_v1', 'label_scalp', 'scalp_signature_v1', 'label_swing', 'swing_signature_v1', 'label_position', 'position_signature_v1', 'ret_1d', 'ret_5d', 'ret_10d', 'ret_20d', 'volume_ratio_20d', 'close_vs_ma20', 'volatility_20d', 'traded_value_proxy', 'buyer_dominance_ratio', 'net_flow_ratio', 'rank1_same_buyer_streak', 'market_regime']


,date,ticker,label_momentum_5d,momentum_5d_signature_v1,label_momentum_10d,momentum_10d_signature_v1,label_momentum_20d,momentum_20d_signature_v1,label_scalp,scalp_signature_v1,...,ret_10d,ret_20d,volume_ratio_20d,close_vs_ma20,volatility_20d,traded_value_proxy,buyer_dominance_ratio,net_flow_ratio,rank1_same_buyer_streak,market_regime
0,2024-12-05,AADI,1.0,0.825643,1.0,0.829718,1.0,0.791360,1,0.749196,...,NaN,NaN,NaN,NaN,NaN,3.055010e+09,0.718460,0.001224,1.0,neutral
1,2024-12-06,AADI,1.0,0.892703,1.0,0.826943,0.0,0.783650,1,0.794775,...,NaN,NaN,NaN,NaN,NaN,3.561635e+09,0.574418,0.000529,1.0,neutral
2,2024-12-09,AADI,0.0,0.994715,0.0,0.994715,0.0,0.905231,1,0.845575,...,NaN,NaN,NaN,NaN,NaN,4.169349e+11,NaN,NaN,1.0,neutral
3,2024-12-10,AADI,0.0,0.879871,0.0,0.887767,0.0,0.858940,0,0.821367,...,NaN,NaN,NaN,NaN,NaN,2.349241e+12,0.296009,0.004887,2.0,neutral
4,2024-12-11,AADI,0.0,0.492591,0.0,0.934044,0.0,0.918408,0,0.645737,...,NaN,NaN,NaN,NaN,NaN,9.621110e+11,0.508773,0.001066,3.0,neutral
